In [1]:
!pip install "pydantic-ai[mcp]"
!pip install "pydantic-ai[fastmcp]"

In [2]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [4]:
!nohup ollama serve > ollama.log 2>&1 &

In [5]:
!ollama run gpt-oss “What is the capital of the Turkey?”

Thinking...
The user asks: "What is the capital of the Turkey?" It's likely they mean "the country Turkey" not "the Turkey". The answer: Ankara. Possibly also mention that Istanbul is largest city. Provide concise answer.
...done thinking.

The capital of Turkey is **Ankara**.



In [12]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) # We get some warnings related to notebook

In [7]:
import asyncio
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.providers.ollama import OllamaProvider

ollama_model = OpenAIChatModel(
model_name='gpt-oss',
provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)

agent = Agent(
    model=ollama_model,
)

In [8]:
result = await agent.run('Write a short poem on computer science')

In [9]:
print(result.output)

**The Binary Muse**

In silicon halls where electrons dance,  
Code whispers in pulses, a silent trance.  
Algorithms rise like constellated stars—  
Logic, art, and the endless quests of ours.  

From loops to recursion, thoughts unwind,  
Machines listening through the data‑spun mind.  
A qubit hums a song of superposed dreams,  
While developers write the future in streams.  

So here, amid the click‑click, we forge and find  
The hidden patterns, the endless lines of code,  
And learn that the deepest algorithm we’ll ever own  
Is the wonder that springs from a curious tone.


In [ ]:
from fastmcp import FastMCP
from pydantic_ai.toolsets.fastmcp import FastMCPToolset


fastmcp_server = FastMCP('my_server')

#Lets make up a code calculation method so that we know our model can't know its own knowledge to guess (we will be sure that it uses the mcp tool)
@fastmcp_server.tool()
async def bixby_code(a: int) -> str:
    return str(10 * a + 55) + "A" * (a%2) + "B" * ((a+1) %2)

toolset = FastMCPToolset(fastmcp_server)

agent = Agent(
    model=ollama_model,
    toolsets=[toolset],
)

In [13]:
result = await agent.run('What is the bixby code of 10?')
print(result.output)

The bixby code for **10** is **155B**.


In [14]:
remote_mcp = FastMCPToolset('https://yoktezmcp.fastmcp.app/mcp')

agent = Agent(
    model=ollama_model,
    toolsets=[remote_mcp],
)

In [15]:
result = await agent.run('What are the thesis names that were advised by Celal Şengör from 2000 to 2025?')
print(result.output)

Here are the thesis titles that were advised by **Celal Şengör** between 2000 and 2025, as retrieved from the YÖK National Thesis Center:

| Year | Thesis Title (English) | Thesis Title (Turkish) |
|------|------------------------|------------------------|
| 2022 |  *An analysis of the knowledge of earth sciences in Middle Byzantine histories, chronicles and military manuals (867‑1204)* |  *Orta Bizans dönemi tarih kaynakları, kronikleri ve askeri talimnamelerindeki yer bilimleri bilgisinin analizi (867‑1204)* |
| 2021 |  *Structure and the tectonic evolution of the Istanbul palaeozoic* |  *İstanbul paleozoyiği'nin yapısı ve tektonik evrimi / Structure and the tectonic evolution of the Istanbul palaeozoic* |
| 2017 |  *Palaeaomagnetic research on the palaeozoic of Istanbul and the place of the latter in hercynides* |  *İstanbul paleozoyiğinin paleomanyetik verilerle incelenmesi ve hersiniyen orojenezindeki yeri / Palaeaomagnetic research on the palaeozoic of Istanbul and the place of t

In [16]:
from pydantic_ai.mcp import MCPServerStreamableHTTP

remote_mcp = MCPServerStreamableHTTP("https://subwayinfo.nyc/mcp")

In [17]:
agent = Agent(
    model=ollama_model,
    toolsets=[remote_mcp],
)

In [18]:
result = await agent.run('Are there Citibikes near Grand Central?')
print(result.output)

Sure! Here’s a list of Citibike stations that are close to Grand Central:

| Station Name | Bikes Available | Docks Available | Distance (approx.) |
|--------------|-----------------|-----------------|--------------------|
| **E 40 St & Park Ave** | 23 | 94 | ~0.2 mi (≈3 min walk) |
| **E 41 St & Park Ave** | 15 | 94 | ~0.3 mi (≈4 min walk) |
| **E 40 St & 5 Ave** | 12 | 81 | ~0.4 mi (≈5 min walk) |
| **E 47 St & Park Ave** | 18 | 93 | ~0.5 mi (≈6 min walk) |

All of these stations have a Citibike kiosk and are located on Broadway, the same block as Grand Central’s exit. If you’d like real‑time availability or directions, just let me know!


In [19]:
yoktez_mcp = FastMCPToolset('https://yoktezmcp.fastmcp.app/mcp')
subway_nyc_mcp = MCPServerStreamableHTTP("https://subwayinfo.nyc/mcp")

agent = Agent(
    model=ollama_model,
    toolsets=[yoktez_mcp,subway_nyc_mcp],
)

In [20]:
result = await agent.run('Are there Citibikes near Grand Central? Also search for the thesis that were advised by Celal Şengör from 2000 to 2025?')
print(result.output)

**Citibike stations near Grand Central (Manhattan) – current snapshot (2026‑01‑28)**  

| Station name | Bikes available | e‑bikes available | Docks available | Capacity | Status |
|--------------|-----------------|-------------------|-----------------|----------|--------|
| Park Ave & E 41 St | 61 | 58 | 42 | 109 | full (operational) |
| E 47 St & Park Ave | 36 | 33 | 72 | 111 | high (operational) |
| E 40 St & 5 Ave | 24 | 19 | 63 | 93 | high (operational) |

These are the only Citibike stations listed by the API that match “Grand Central” within Manhattan. Each station currently reports bikes, e‑bikes, dock availability, and is operational.

---

**Theses advised by Celal Şengör (2000 – 2025) – YÖK National Thesis Center**  

| Year | Thesis No. | Title | Author | PhD/Master | University |
|------|-----------|-------|--------|------------|------------|
| 2022 | 707336 | *An analysis of the knowledge of earth sciences in Middle Byzantine histories, chronicles and military manuals (86

In [21]:
result = await agent.run('Write a short poem on tectonic and magmatic structure and NYC railways')
print(result.output)

**Whispers of Earth and Steel**

Beneath the bedrock's restless spine,  
Tectonic plates in hush align—  
A ballet of plates, a quiet churn,  
Where magma brews and continents learn.

Stroking the crust with molten fire,  
The earth's heartbeat never tires,  
In fissures deep, the ash is born,  
A silent promise, a volcanic morn.

Above, the subway rivers flow,  
Rails etched through the city's glow,  
Like veins of steel beneath the streets,  
Binding boroughs, humming beats.

From granite roots to iron veins,  
Both worlds converge in city pains—  
The earth's secret, the rails' bright song,  
A harmony of deep and long.
